# Milestone 2 — Lakebase Operational Schema & Writable Tables (EXECUTED)

Executed on Databricks serverless against the Lakebase Postgres DB via UC federation (`streamline_lakebase`). Run: https://adb-7405612117836809.9.azuredatabricks.net/?o=7405612117836809#job/282089609308916/run/789290406925902

Sync defined in code: `resources/lakebase_sync.yml` (DAB `PostgresSyncedTable`). Schema in `migrations/00[1-3]_*.sql`.

## 1. Operational schema — the four related tables exist and are populated

In [1]:
spark.sql("""SELECT ... COUNT(*) FROM streamline_lakebase.public.care_agents UNION ALL ...""").show()

operational_table  | rows
-------------------+-----
care_action_events | 4   
care_actions       | 4   
care_agents        | 1   
care_offer_catalog | 5   


## 2. Related tables JOIN — agent → action → offer → lifecycle events
Proves the operational schema joins end-to-end across its foreign keys.

In [1]:
spark.sql("""SELECT ag.display_name, ca.subscriber_id, oc.offer_name, ca.status, ... FROM care_actions ca JOIN care_agents ag ... LEFT JOIN care_offer_catalog oc ... LEFT JOIN care_action_events ev ...""").show()

agent             | subscriber_id | offer_name               | status   | predicted_retained_clv_usd | lifecycle_events
------------------+---------------+--------------------------+----------+----------------------------+-----------------
Mathieu Pelletier | SUB-0000214   | One-Time Bill Credit $50 | executed | 1357.19                    | 1               
Mathieu Pelletier | SUB-0000301   | One-Time Bill Credit $50 | executed | 1364.53                    | 1               
Mathieu Pelletier | SUB-0000412   | One-Time Bill Credit $50 | approved | 1312.44                    | 1               
Mathieu Pelletier | SUB-0007220   | One-Time Bill Credit $50 | executed | 1301.55                    | 1               


## 3. Writable operational tables vs read-only synced mirrors

In [1]:
spark.sql("""SELECT relation, role, COUNT(*) ... (writable care_* vs synced mirrors)""").show()

relation                  | role                   | rows 
--------------------------+------------------------+------
offers                    | READ-ONLY synced (DAB) | 5    
open_atrisk               | READ-ONLY synced (DAB) | 200  
retention_recommendations | READ-ONLY synced (DAB) | 200  
subscriber_position       | READ-ONLY synced (DAB) | 40000
care_action_events        | WRITABLE               | 4    
care_actions              | WRITABLE               | 4    
care_agents               | WRITABLE               | 1    
care_offer_catalog        | WRITABLE               | 5    


## Result
`dbutils.notebook.exit(...)` payload:

In [1]:
print(json.dumps(result, indent=2))

{
  "operational_schema": "verified",
  "sync_defined_in": "resources/lakebase_sync.yml (DAB PostgresSyncedTable)"
}